1️⃣ Architecture Overview
User
  ↓
OpenAI Model (Agent)
  ↓
MCP Client (Python)
  ↓
MCP Server (Tool Provider)
  ↓
MySQL Database

CREATE DATABASE company;

USE company;

CREATE TABLE employees (
    id INT PRIMARY KEY AUTO_INCREMENT,
    name VARCHAR(100),
    department VARCHAR(100),
    salary INT
);

INSERT INTO employees (name, department, salary) VALUES
('Amit Sharma', 'Engineering', 120000),
('Priya Verma', 'Engineering', 95000),
('Rahul Gupta', 'Marketing', 80000),
('Sneha Patil', 'HR', 70000),
('Vikram Singh', 'Engineering', 150000),
('Neha Kulkarni', 'Finance', 110000),
('Arjun Mehta', 'Sales', 90000),
('Pooja Nair', 'Marketing', 85000),
('Rohan Deshmukh', 'Engineering', 105000),
('Kavita Joshi', 'HR', 72000);

Python MCP Tool for MySQL

In [ ]:
!pip install openai mysql-connector-python fastapi uvicorn mysql 
!pip install FastMCP

In [ ]:
import sys
import os

print("Python version:", sys.version)
print("Python executable:", sys.executable)

try:
    from mcp.server.fastmcp import FastMCP
    print("✅ FastMCP is now available!")
except ImportError as e:
    print("❌ FastMCP module not found.")
    print("Error:", e)
    print("\nTry these steps:")
    print("1. Install MCP: pip install mcp")
    print("2. Restart the kernel or Python environment")
    print("3. Verify installation using: pip show mcp")

import mysql.connector
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("mysql-server")

def run_query(query: str):
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="",
        database="company"
    )

    cursor = conn.cursor(dictionary=True)
    cursor.execute(query)

    result = cursor.fetchall()

    cursor.close()
    conn.close()

    return result


@mcp.tool()
def query_mysql(query: str):
    """Execute a SQL query on MySQL database"""
    return run_query(query)


if __name__ == "__main__":
    mcp.run()

Python Client Using OpenAI + MCP

In [ ]:
# Cell 2: Create the server code (weather.py)
# We'll write the server file to disk so the subprocess can run it. host="127.0.0.1", port=8000"
import os
port = int(os.getenv("MCP_PORT", 9000))

server_code = """
from fastmcp import FastMCP
import mysql.connector

app = FastMCP("mysqlserver")

def run_query(query: str):
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="",
        database="company"
    )
    cursor = conn.cursor(dictionary=True)
    cursor.execute(query)
    result = cursor.fetchall()
    cursor.close()
    conn.close()
    return result

@app.tool()
def query_mysql(query: str):
    return run_query(query)

if __name__ == "__main__":
    # For stdio transport (works with stdio_client)
    app.run(transport="sse", host="localhost", port=9000)


    # If you want SSE transport, remove the line above and instead run:
    # app.run(transport="sse")
    # Then start with: uvicorn employeeserver:app --host 127.0.0.1 --port 8000
"""

with open("employeeserver.py", "w") as f:
    f.write(server_code)

print("employeeserver.py written")

In [ ]:
!python employeeserver.py

In [ ]:
from mcp.client.sse import sse_client
from mcp.client.session import ClientSession

async def main():
    async with sse_client("http://127.0.0.1:8000/sse") as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print("Tools:", tools)

            result = await session.call_tool("query_mysql", {"query": "SELECT * FROM employees"})
            print("Result:", result)

await main()